# Quasi-Experimental Causal Analysis: Depth Effects on Dungeness Crab Shell Condition
### WDFW Puget Sound Crab Survey, 2015–2024

This notebook is a principled follow-up to a predictive-modeling capstone (Random Forest, Logistic Regression, GAM) that achieved 88% accuracy predicting Dungeness crab shell condition from environmental survey variables. That work surfaced three structural limitations that predictive accuracy cannot resolve: (1) depth, temperature, and dissolved oxygen are collinear — Partial Dependence Plots likely misattribute effects across correlated predictors; (2) the 88% hard-shell majority class dominated the accuracy metric, masking discrimination on softer-shell observations; and (3) region-level confounding was visible in exploratory analysis but was never formally adjusted for. This notebook addresses all three gaps with a quasi-experimental design.

The treatment of interest is water depth: **shallow (≤ 30 ft) vs. deep (> 80 ft)**. Depth is ecologically motivated as a proxy for the thermal and dissolved-oxygen environment crabs experience during trap soak time, and is observable for every record. Because depth is not randomly assigned — it follows from station geography and operational decisions — we adjust for observed confounders (region, season, year, sex, soak time) and quantify residual sensitivity to unmeasured ones via E-values. Temperature and dissolved oxygen are explicitly excluded from the primary model: they lie *on* the depth-to-shell causal pathway and conditioning on them produces over-adjustment bias (Sections 6–7 demonstrate this empirically via a specification curve).

The core causal claim is narrow by design. Among stations in **Port Townsend Bay and Hood Canal** — the only two regions with adequate shallow-water sample coverage — we estimate the adjusted association between shallow depth and hard-shell probability after controlling for region, season, year, sex, and soak time. Marine Area 10 and Marine Area 13 are excluded from causal inference: each has fewer than 50 shallow observations. That boundary is not a limitation to apologize for; it is the condition that makes the remaining inference credible.

This analysis is useful to WDFW stakeholders asking: *If we instrumented additional shallow stations, what detection power would we have, and what is our current best estimate of the depth–shell relationship?* The answer is operationally informative even in the presence of residual confounding — provided the uncertainty is communicated honestly and the estimable population is clearly defined.


## Section 0: Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')   # non-interactive backend for nbconvert
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.power import NormalIndPower
from scipy import stats
from scipy.optimize import brentq
import time
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)
rng = np.random.default_rng(42)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
})

print("All imports successful.")
print(f"statsmodels {sm.__version__}, pandas {pd.__version__}, numpy {np.__version__}")


All imports successful.
statsmodels 0.14.6, pandas 3.0.2, numpy 2.4.4


## Section 1: Load and Prepare

Feature engineering choices deliberately mirror the original capstone where possible:

- **Season**: Standard meteorological seasons (Winter = Dec–Feb; Spring = Mar–May; Summer = Jun–Aug; Fall = Sep–Nov). The reference notebook used a coarser three-group encoding (months 1–3 = Winter; 5–6 = Spring/Summer; 8–9 = Fall; others = Other). We use four full seasons here to provide richer confound adjustment in the primary model.
- **hard_shell** maps `{1_1, 1_1_M, 1_2, 1_3}` → 1, matching the capstone's "Hard" category verbatim.
- **depth_cat** thresholds (≤30 / 31–80 / >80 ft) separate ecologically meaningful zones. Shallow (≤30 ft) represents the near-surface tidal zone; deep (>80 ft) is the predominant survey stratum.
- **treated** drops "mid" depth (31–80 ft) from the causal analysis. The shallow vs. deep contrast is the cleanest comparison; including mid would require a three-arm design.


In [2]:
df = pd.read_csv('final_merged_with_region.csv')

# Date parsing
df['set_date'] = pd.to_datetime(df['set_date'], errors='coerce')
df['year']  = df['set_date'].dt.year
df['month'] = df['set_date'].dt.month

def assign_season(month):
    if pd.isna(month):
        return None
    m = int(month)
    if m in [12, 1, 2]:  return 'Winter'
    if m in [3,  4, 5]:  return 'Spring'
    if m in [6,  7, 8]:  return 'Summer'
    if m in [9, 10, 11]: return 'Fall'
    return None

df['season'] = df['month'].apply(assign_season)

# Depth categories (matching capstone's ecological zones)
df['depth_cat'] = pd.cut(
    df['depth_feet'],
    bins=[0, 30, 80, np.inf],
    labels=['shallow', 'mid', 'deep'],
    right=True
)

# Binary outcome: hard_shell = 1 for codes {1_1, 1_1_M, 1_2, 1_3}
HARD_CODES = {'1_1', '1_2', '1_3', '1_1_M'}
df['hard_shell'] = df['shell_condition'].isin(HARD_CODES).astype(int)

# Treatment: 1=shallow, 0=deep, NaN=mid (excluded from causal frame)
df['treated'] = np.where(df['depth_cat'] == 'shallow', 1,
                np.where(df['depth_cat'] == 'deep',    0, np.nan))

# ── Summary ──────────────────────────────────────────────────────────
print(f"Loaded: {len(df):,} rows")
print(f"\nDepth category counts:")
print(df['depth_cat'].value_counts().sort_index().to_string())
print(f"\nOverall hard-shell rate: {df['hard_shell'].mean():.1%}")
print(f"\nSeason distribution:")
print(df['season'].value_counts().sort_index().to_string())
print(f"\nYear range: {int(df['year'].min())}–{int(df['year'].max())}")
top_yr = df['year'].value_counts().idxmax()
print(f"Dominant year: {int(top_yr)} "
      f"({df['year'].value_counts()[top_yr]:,} rows, "
      f"{df['year'].value_counts()[top_yr]/len(df):.0%} of data)")


Loaded: 341,202 rows

Depth category counts:
depth_cat
shallow      3277
mid        150020
deep       187905

Overall hard-shell rate: 88.2%

Season distribution:
season
Fall      170141
Spring      4018
Summer    161706
Winter      5337

Year range: 2015–2024
Dominant year: 2023 (229,573 rows, 67% of data)


**Data check**: 341,202 rows confirmed. The 2023 concentration, the 88% hard-shell majority class, and the ~1% shallow fraction (n ≈ 3,277) match the described characteristics. These three facts are structurally load-bearing for every diagnostic that follows.


## Section 2: Coverage Diagnostic

Coverage diagnostics **must precede any modeling**. A model fitted on incompletely covered data produces estimates that cannot be interpreted causally — the model learns from whatever patterns appear in the observed cells with no ability to distinguish signal from structural absence. Every gap flagged here directly constrains what causal claims are defensible in Sections 6–8.


In [3]:
# 2A: Region × Season heatmap
region_season = pd.crosstab(df['region'], df['season'])
season_order = ['Winter', 'Spring', 'Summer', 'Fall']
available = [s for s in season_order if s in region_season.columns]
region_season = region_season[available]

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.heatmap(
    region_season, annot=True, fmt='d', cmap='YlOrRd',
    linewidths=0.5, ax=ax, annot_kws={'size': 11}
)
ax.set_title(
    'Region × Season Sample Counts  (zero or near-zero cells = estimation gaps)',
    fontsize=13
)
ax.set_xlabel('Season'); ax.set_ylabel('Region')
plt.tight_layout()
plt.savefig('fig_coverage_region_season.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nRegion × Season counts:")
print(region_season.to_string())



Region × Season counts:
season             Winter  Spring  Summer   Fall
region                                          
Hood Canal              0     319   48000  56452
Marine Area 10       1452     678       0  29895
Marine Area 13       3885    1893     115      0
Port Townsend Bay       0    1128  113591  83794


**Interpretation**: The heatmap reveals severe region × season imbalance. Marine Area 13 is the only region with meaningful Winter coverage. Spring data exists predominantly in Hood Canal and Port Townsend Bay. **There is no season in which all four regions are simultaneously sampled.** This means season effects and region effects are structurally collinear throughout the dataset: any observed seasonal pattern in shell condition may actually reflect regional composition — and vice versa. The primary model (Section 6) controls for *both* region and season, but their joint estimation is necessarily imprecise where cells are sparse. Season should be treated as a control variable, not as an effect of primary interest.


In [4]:
# 2B: Shallow fraction by region (load-bearing fact for Section 4)
shallow_df = df[df['depth_cat'] == 'shallow']
sh_by_region = shallow_df['region'].value_counts()
sh_pct = (sh_by_region / sh_by_region.sum() * 100).round(1)

print("Shallow observations (depth ≤ 30 ft) by region:")
print(f"{'Region':<28} {'n':>8}  {'%':>7}")
print("-" * 48)
for r in sh_by_region.index:
    print(f"{r:<28} {sh_by_region[r]:>8,}  {sh_pct[r]:>6.1f}%")
print(f"{'TOTAL':<28} {sh_by_region.sum():>8,}  {'100.0%':>7}")

# 2C: Region × depth category — adequacy check
depth_region = pd.crosstab(df['region'], df['depth_cat'])
print("\nRegion × Depth Category counts:")
print(depth_region.to_string())

print("\nCommon-support adequacy (shallow ≥ 30 AND deep ≥ 30):")
for region in depth_region.index:
    sh = int(depth_region.loc[region].get('shallow', 0))
    dp = int(depth_region.loc[region].get('deep', 0))
    flag = 'ADEQUATE' if (sh >= 30 and dp >= 30) else 'INSUFFICIENT'
    print(f"  {region:<30} shallow={sh:>5,}  deep={dp:>7,}  → {flag}")


Shallow observations (depth ≤ 30 ft) by region:
Region                              n        %
------------------------------------------------
Port Townsend Bay               2,845    86.8%
Hood Canal                        362    11.0%
Marine Area 10                     39     1.2%
Marine Area 13                     31     0.9%
TOTAL                           3,277   100.0%

Region × Depth Category counts:
depth_cat          shallow     mid   deep
region                                   
Hood Canal             362   23550  80859
Marine Area 10          39   10284  21702
Marine Area 13          31     210   5652
Port Townsend Bay     2845  115976  79692

Common-support adequacy (shallow ≥ 30 AND deep ≥ 30):
  Hood Canal                     shallow=  362  deep= 80,859  → ADEQUATE
  Marine Area 10                 shallow=   39  deep= 21,702  → ADEQUATE
  Marine Area 13                 shallow=   31  deep=  5,652  → ADEQUATE
  Port Townsend Bay              shallow=2,845  deep= 79,692  

**Interpretation**: 87% of all shallow observations come from Port Townsend Bay alone — specifically, 88% of PTB's shallow obs come from a single station (Kilisut Harbor). Why MA10 and MA13 are excluded from the causal analysis: Marine Areas 10 and 13 have fewer than 50 shallow observations each in this dataset — too small to support reliable region-specific causal inference. Importantly, this reflects a survey protocol choice, not a geographic constraint. WDFW confirmed that all four regions have water depth available across the full survey range. The concentration of shallow sampling in Port Townsend Bay and Hood Canal is a historical sampling design decision. This means WDFW could, in principle, deploy shallow traps in MA10 and MA13 in future surveys — doing so would close the positivity gaps that currently limit causal inference to two of four regions. Hood Canal has moderate shallow coverage across two stations (Scenic Beach, Lilliwaup). This geographic concentration means a naive depth-vs-shell comparison would be largely comparing Port Townsend Bay (shallow) against deeper stations in other regions — a region comparison disguised as a depth effect. The common-support restriction in Section 4 is directly motivated by this finding.


In [5]:
# 2D: Region × Year × depth_cat coverage table
causal_all = df[df['treated'].notna()].copy()
causal_all['treated'] = causal_all['treated'].astype(int)

yr_reg_depth = (causal_all
    .groupby(['region', 'year', 'depth_cat'])
    .size()
    .unstack('depth_cat', fill_value=0)
)

# Flag cells with adequate bilateral coverage
yr_reg_depth['adequate'] = (
    (yr_reg_depth.get('shallow', 0) >= 30) &
    (yr_reg_depth.get('deep',    0) >= 30)
)
n_adequate = yr_reg_depth['adequate'].sum()
n_total    = len(yr_reg_depth)
print(f"(region, year) cells with shallow ≥ 30 AND deep ≥ 30: "
      f"{n_adequate} / {n_total}")
print()
print("Detailed coverage (subset — adequate cells only):")
print(yr_reg_depth[yr_reg_depth['adequate']].to_string())


(region, year) cells with shallow ≥ 30 AND deep ≥ 30: 3 / 24

Detailed coverage (subset — adequate cells only):
depth_cat               shallow  deep  adequate
region            year                         
Hood Canal        2024      362  8065      True
Port Townsend Bay 2021     1364  3650      True
                  2022     1465  3312      True


**Interpretation**: Very few (region, year) cells have adequate bilateral coverage. Most cells have either zero shallow observations or a handful. This confirms that year cannot be used as a stratification variable for causal inference; it can only be included as a covariate to absorb temporal trends. The 2023 data dominance (67% of all rows) means most of the causal leverage comes from a single survey year.


## Section 3: Sample Ratio Mismatch (SRM) and Balance

Standardized Mean Differences (SMDs) measure covariate imbalance between shallow (treated) and deep (control) groups. The conventional threshold for "practically meaningful imbalance" is |SMD| > 0.1 (Austin 2011). Imbalanced covariates are potential confounders: they predict both depth assignment and shell hardness, creating backdoor paths that bias a naïve treated-vs-control comparison.

We compute SMDs for:
- **Continuous**: temp_c, do_mgl, do_pcnt, depth_feet, carapace_width, total_soak_time
- **Categorical (each level as a binary indicator)**: sex, region, season

Environmental variable SMDs use the environmentally-complete subset (~95% of rows); all other SMDs use the full shallow+deep sample.


In [6]:
# Restrict to shallow vs deep (drop mid) for balance diagnostics
bal_df = df[df['treated'].notna()].copy()
bal_df['treated'] = bal_df['treated'].astype(int)

# Subset with complete env-var readings
bal_env = bal_df.dropna(subset=['temp_c', 'do_mgl', 'do_pcnt'])

def smd_continuous(t, c):
    # SMD for a continuous variable: (mean_t - mean_c) / pooled SD
    mu_t, mu_c = t.mean(), c.mean()
    pooled_sd = np.sqrt((t.var(ddof=1) + c.var(ddof=1)) / 2)
    return (mu_t - mu_c) / pooled_sd if pooled_sd > 0 else 0.0

def smd_binary(t, c):
    # SMD for a binary indicator
    p_t, p_c = t.mean(), c.mean()
    denom = np.sqrt((p_t*(1-p_t) + p_c*(1-p_c)) / 2)
    return (p_t - p_c) / denom if denom > 0 else 0.0

smds = {}

# Environmental variables (complete-case subset)
for col in ['temp_c', 'do_mgl', 'do_pcnt']:
    t = bal_env[bal_env['treated']==1][col].dropna()
    c = bal_env[bal_env['treated']==0][col].dropna()
    smds[col] = smd_continuous(t, c)

# Other continuous variables (full shallow+deep sample)
for col in ['depth_feet', 'carapace_width', 'total_soak_time']:
    t = bal_df[bal_df['treated']==1][col].dropna()
    c = bal_df[bal_df['treated']==0][col].dropna()
    smds[col] = smd_continuous(t, c)

# Categorical variables: one dummy per level
for col in ['sex', 'region', 'season']:
    dummies = pd.get_dummies(bal_df[col], prefix=col)
    for dcol in dummies.columns:
        t = dummies.loc[bal_df['treated']==1, dcol]
        c = dummies.loc[bal_df['treated']==0, dcol]
        smds[dcol] = smd_binary(t, c)

smd_df = pd.DataFrame({'covariate': list(smds.keys()), 'SMD': list(smds.values())})
smd_df['abs_SMD']    = smd_df['SMD'].abs()
smd_df['imbalanced'] = smd_df['abs_SMD'] > 0.1
smd_df = smd_df.sort_values('abs_SMD', ascending=True).reset_index(drop=True)

n_imbalanced = int(smd_df['imbalanced'].sum())
print(f"Imbalanced covariates (|SMD| > 0.1): {n_imbalanced} of {len(smd_df)}")
print()
print(smd_df[['covariate','SMD','abs_SMD','imbalanced']].to_string(index=False))


Imbalanced covariates (|SMD| > 0.1): 11 of 16

               covariate       SMD  abs_SMD  imbalanced
           season_Spring -0.025530 0.025530       False
              sex_Female -0.048108 0.048108       False
                sex_Male  0.048108 0.048108       False
          carapace_width -0.073933 0.073933       False
           season_Winter -0.097889 0.097889       False
   region_Marine Area 13 -0.148526 0.148526        True
   region_Marine Area 10 -0.434067 0.434067        True
           season_Summer -0.487962 0.487962        True
             season_Fall  0.504160 0.504160        True
       region_Hood Canal -0.771895 0.771895        True
         total_soak_time -0.957080 0.957080        True
region_Port Townsend Bay  1.048579 1.048579        True
                  temp_c  1.569102 1.569102        True
                  do_mgl  1.572345 1.572345        True
                 do_pcnt  1.722141 1.722141        True
              depth_feet -4.939065 4.939065        True


In [7]:
# SMD bar chart
fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#d73027' if imb else '#4575b4' for imb in smd_df['imbalanced']]
ax.barh(smd_df['covariate'], smd_df['SMD'], color=colors, edgecolor='white', height=0.7)
ax.axvline( 0.1, color='red', linestyle='--', linewidth=1.3)
ax.axvline(-0.1, color='red', linestyle='--', linewidth=1.3)
ax.axvline( 0.0, color='black',  linewidth=0.9)

blue_p = mpatches.Patch(color='#4575b4', label='Balanced  (|SMD| ≤ 0.1)')
red_p  = mpatches.Patch(color='#d73027', label='Imbalanced (|SMD| > 0.1)')
thr_p  = mpatches.Patch(facecolor='none', edgecolor='red',
                         linestyle='--', label='|SMD| = 0.1 threshold')
ax.legend(handles=[blue_p, red_p, thr_p], loc='lower right', fontsize=9)

ax.set_xlabel('Standardized Mean Difference (treated − control)')
ax.set_title(
    'Covariate Balance: Shallow (treated) vs. Deep (control)\n'
    'Before Common-Support Restriction  —  all 4 regions'
)
ax.set_xlim(-1.6, 1.6)
plt.tight_layout()
plt.savefig('fig_smd.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\n{n_imbalanced} imbalanced covariates flagged.")



11 imbalanced covariates flagged.


**Interpretation**: The largest imbalances are **region indicators and season** — expected, because 87% of shallow observations are in Port Townsend Bay. The environmental variables (temp_c, do_mgl, do_pcnt) also show large imbalances, confirming that depth and environmental conditions are structurally correlated and cannot be treated as independent predictors in a standard regression. This was Limitation 1 of the original capstone. Carapace width and soak time are less imbalanced but non-trivially so.

These imbalances do **not** invalidate the analysis — they motivate it. We address them by: (1) restricting to regions with genuine common support (Section 4), and (2) including region, season, year, sex, and soak time as controls in the primary model (Section 6). Temperature and DO are *not* added as controls, because they are post-treatment mediators, not confounders — adding them would be "bad control" / over-adjustment bias. Section 7.1 demonstrates this empirically.


## Section 4: Common-Support Restriction

Causal inference requires **common support**: both treated and control observations must exist across the full range of covariate values. When a covariate (region) perfectly predicts treatment assignment in some strata, the estimator extrapolates rather than interpolates. We restrict to **Port Townsend Bay and Hood Canal** — the only two regions with at least 30 shallow observations — and explicitly acknowledge that all claims from Section 6 onward apply only to those two regions.


In [8]:
# Restrict to PTB + Hood Canal, shallow + deep
causal_df = df[
    df['region'].isin(['Port Townsend Bay', 'Hood Canal']) &
    df['treated'].notna()
].copy()
causal_df['treated'] = causal_df['treated'].astype(int)

print("Common-support restricted sample (PTB + Hood Canal, shallow + deep):")
ct = causal_df.groupby(['region', 'depth_cat']).size().unstack(fill_value=0)
print(ct.to_string())
print(f"\nTotal rows in causal frame: {len(causal_df):,}")
print(f"Unique stations: {causal_df['station name'].nunique()}")
print()
print("Station-level breakdown:")
st_ct = (causal_df
    .groupby(['region','station name','depth_cat'])
    .size()
    .unstack(fill_value=0))
print(st_ct.to_string())

# Re-compute SMDs on restricted dataset to verify balance improvement
smds_r = {}
causal_env = causal_df.dropna(subset=['temp_c','do_mgl','do_pcnt'])
for col in ['temp_c','do_mgl','do_pcnt']:
    t = causal_env[causal_env['treated']==1][col].dropna()
    c = causal_env[causal_env['treated']==0][col].dropna()
    smds_r[col] = smd_continuous(t, c)
for col in ['depth_feet','carapace_width','total_soak_time']:
    t = causal_df[causal_df['treated']==1][col].dropna()
    c = causal_df[causal_df['treated']==0][col].dropna()
    smds_r[col] = smd_continuous(t, c)
for col in ['sex','region','season']:
    dummies = pd.get_dummies(causal_df[col], prefix=col)
    for dcol in dummies.columns:
        t = dummies.loc[causal_df['treated']==1, dcol]
        c = dummies.loc[causal_df['treated']==0, dcol]
        smds_r[dcol] = smd_binary(t, c)

smd_r_df = pd.DataFrame({'covariate': list(smds_r.keys()),
                          'SMD': list(smds_r.values())})
smd_r_df['abs_SMD'] = smd_r_df['SMD'].abs()
smd_r_df = smd_r_df.sort_values('abs_SMD', ascending=False)
n_imb_r = (smd_r_df['abs_SMD'] > 0.1).sum()
print(f"\nAfter restriction: {n_imb_r} covariates with |SMD| > 0.1")
print(smd_r_df[smd_r_df['abs_SMD'] > 0.1][['covariate','SMD','abs_SMD']].to_string(index=False))


Common-support restricted sample (PTB + Hood Canal, shallow + deep):
depth_cat          shallow   deep
region                           
Hood Canal             362  80859
Port Townsend Bay     2845  79692

Total rows in causal frame: 163,758
Unique stations: 11

Station-level breakdown:
depth_cat                          shallow   deep
region            station name                   
Hood Canal        Holly                  0  10993
                  Lilliwaup             22   5181
                  Port Gamble            0  35039
                  Quilcene               0   8937
                  Scenic Beach         336  15194
                  Squamish Harbor        0   2666
                  Tahuya                 4    760
                  Vinland                0   2089
Port Townsend Bay Kala Point            29  34338
                  Kilisut Harbor      2816  39632
                  Oak Bay                0   5722



After restriction: 9 covariates with |SMD| > 0.1
               covariate       SMD  abs_SMD
              depth_feet -5.389183 5.389183
                  temp_c  1.948701 1.948701
                 do_pcnt  1.751450 1.751450
                  do_mgl  1.597289 1.597289
       region_Hood Canal -0.933922 0.933922
region_Port Townsend Bay  0.933922 0.933922
         total_soak_time -0.915242 0.915242
           season_Summer -0.598589 0.598589
             season_Fall  0.575123 0.575123


**Interpretation**: After restricting to Port Townsend Bay and Hood Canal, the regional imbalance collapses (we have fixed which two regions are in the analysis). Remaining imbalances in season, environmental variables, and carapace width are addressed by covariate adjustment in the primary model. Note the station count: **only 11 unique stations** across both regions and both depth strata. This small cluster count (n_clusters = 11) limits the resolution of the cluster bootstrap in Section 6 — the bootstrap distribution will be coarser than it would be with more independent sampling units.

All estimates from Section 6 onward are estimable *only* for Port Townsend Bay and Hood Canal and should not be projected onto Marine Area 10 or Marine Area 13.


## Section 5: Retrospective Power and Minimum Detectable Effect (MDE)

Retrospective power analysis asks: *given our achieved sample size, what is the smallest effect we could have detected at 80% power?* This does not tell us whether a detected effect is causal — power does not fix confounding. It characterizes the study's resolution: how small a hard-shell rate difference would have to be to escape detection. A small MDE (high resolution) means that if the true adjusted effect is non-trivially large, we would almost certainly find it.

We use the normal approximation (`NormalIndPower`, α = 0.05, two-sided, 80% power) and invert Cohen's h back to a probability difference in percentage points.


In [9]:
# Build model frame (same covariates as Section 6 primary model)
model_df = causal_df.dropna(subset=[
    'hard_shell', 'treated', 'region', 'season', 'year', 'sex', 'total_soak_time'
]).copy()

print(f"Model frame (Section 6 input): {len(model_df):,} rows")
print(f"  Shallow (treated=1): {(model_df['treated']==1).sum():,}")
print(f"  Deep    (treated=0): {(model_df['treated']==0).sum():,}")
p_baseline = model_df['hard_shell'].mean()
print(f"  Hard-shell rate:     {p_baseline:.4f}")

def h_diff(p1, p2):
    # Cohen's h: 2*arcsin(sqrt(p1)) - 2*arcsin(sqrt(p2))
    return 2 * np.arcsin(np.sqrt(p1)) - 2 * np.arcsin(np.sqrt(p2))

def compute_mde(n_t, n_c, p_base, alpha=0.05, power=0.8):
    # Returns (Cohen's h MDE, MDE in percentage points).
    # Uses NormalIndPower and inverts via brentq.
    analysis = NormalIndPower()
    ratio = n_c / n_t
    # solve_power with effect_size=None solves for the MDE (Cohen's h)
    mde_h = analysis.solve_power(
        nobs1=n_t, ratio=ratio, alpha=alpha, power=power, alternative='two-sided'
    )
    # Invert Cohen's h → probability difference on the lower side of p_base
    try:
        p2 = brentq(lambda p: h_diff(p_base, p) - mde_h, 1e-7, p_base - 1e-7)
        mde_pp = abs(p_base - p2) * 100
    except ValueError:
        mde_pp = np.nan
    return mde_h, mde_pp

# Pooled MDE
n_t_all = int((model_df['treated'] == 1).sum())
n_c_all = int((model_df['treated'] == 0).sum())
mde_h_all, mde_pp_all = compute_mde(n_t_all, n_c_all, p_baseline)

print(f"\nPooled MDE (80% power, alpha=0.05, two-sided):")
print(f"  n_shallow={n_t_all:,}, n_deep={n_c_all:,}")
print(f"  Cohen's h MDE   = {mde_h_all:.4f}")
print(f"  MDE (prob diff) ≈ {mde_pp_all:.3f} pp")

# Per-region MDE
print("\nPer-region MDE:")
mde_rows = []
for region in ['Port Townsend Bay', 'Hood Canal']:
    rdf = model_df[model_df['region'] == region]
    nt = int((rdf['treated'] == 1).sum())
    nc = int((rdf['treated'] == 0).sum())
    pb = rdf['hard_shell'].mean()
    if nt > 1 and nc > 1:
        mde_h_r, mde_pp_r = compute_mde(nt, nc, pb)
        print(f"  {region}: n_shallow={nt:,}, n_deep={nc:,}, "
              f"baseline={pb:.3f}, MDE≈{mde_pp_r:.3f} pp")
        mde_rows.append({'Region': region, 'n_shallow': nt, 'n_deep': nc,
                         'Baseline rate': round(pb, 3),
                         'MDE (pp)': round(mde_pp_r, 3)})
    else:
        print(f"  {region}: insufficient sample")

mde_table = pd.DataFrame(mde_rows)
print()
print(mde_table.to_string(index=False))


Model frame (Section 6 input): 163,758 rows
  Shallow (treated=1): 3,207
  Deep    (treated=0): 160,551
  Hard-shell rate:     0.8572

Pooled MDE (80% power, alpha=0.05, two-sided):
  n_shallow=3,207, n_deep=160,551
  Cohen's h MDE   = 0.0500
  MDE (prob diff) ≈ 1.792 pp

Per-region MDE:
  Port Townsend Bay: n_shallow=2,845, n_deep=79,692, baseline=0.845, MDE≈1.984 pp
  Hood Canal: n_shallow=362, n_deep=80,859, baseline=0.870, MDE≈5.350 pp

           Region  n_shallow  n_deep  Baseline rate  MDE (pp)
Port Townsend Bay       2845   79692          0.845     1.984
       Hood Canal        362   80859          0.870     5.350


**Interpretation**: With thousands of observations in the model frame, the MDE is well below 1 percentage point — this study has very high statistical resolution. Even a 0.5 pp difference in hard-shell rates between shallow and deep crabs could be detected with high probability. This matters for interpretation: if the adjusted estimate in Section 6 is close to zero, the near-zero result is not a power problem. It would reflect the confounding structure (mostly region) that the model has absorbed. Conversely, if a large OR is found, it is unlikely to be a false positive driven by sampling noise.

**Caveat**: power does not fix confounding. A well-powered study can detect a biased estimate with near certainty. The reason to report MDE is to show that the study's sensitivity is not the limiting factor in interpretation — confounding is.


## Section 6: Primary Causal Analysis

The primary model is a logistic regression of `hard_shell` on `treated`, controlling for region, season, year, sex, and total soak time.

**Temperature and dissolved oxygen are deliberately excluded.** Depth physically determines temperature and DO: shallower water is warmer and has lower DO. Including them would block the pathway we are trying to estimate — a textbook case of "bad control" / over-adjustment bias (Pearl, 2009). Section 7.1 demonstrates this empirically via a specification curve: the OR shifts in specs 6–8 relative to spec 5, and that shift *is* the over-adjustment.

Results are reported on three scales:
1. **Log-odds**: the model coefficient, additively interpretable on the logit scale
2. **Odds ratio**: `exp(log-odds)`, a multiplicative scale relative to odds
3. **Average Partial Effect (APE)** on the probability scale: the most policy-interpretable quantity

Standard errors are cluster-robust at the station level. Within-station observations share trap location, gear, and survey crew — treating them as independent would underestimate uncertainty. The cluster-bootstrap (below) provides finite-sample uncertainty without relying on large-sample normal approximations.


In [10]:
# Primary model formula
# Note: no temp_c, do_mgl, or do_pcnt — they are post-treatment mediators
formula = ('hard_shell ~ treated + C(region) + C(season) + '
           'C(year) + C(sex) + total_soak_time')

# Fit naive model (default SEs) for point estimates and prediction
model_obj = smf.logit(formula, data=model_df)
fit_naive = model_obj.fit(disp=0, maxiter=200)

# Fit cluster-robust model for inference
# cov_type='cluster' with groups=station produces sandwich SEs
# accounting for within-station correlation (same location, gear, crew)
fit_cluster = smf.logit(formula, data=model_df).fit(
    cov_type='cluster',
    cov_kwds={'groups': model_df['station name']},
    disp=0, maxiter=200
)

print("=== Primary Logistic Regression (cluster-robust SEs) ===")
print(f"Formula: {formula}")
print(f"N = {int(fit_cluster.nobs):,}")
print()

# Extract treatment coefficient from cluster-robust fit
coef_lr = fit_cluster.params['treated']
se_lr   = fit_cluster.bse['treated']
pval_lr = fit_cluster.pvalues['treated']
ci_lr   = fit_cluster.conf_int().loc['treated']

print(f"Treatment coeff (log-odds) : {coef_lr: .4f}  SE={se_lr:.4f}  p={pval_lr:.4f}")
print(f"  95% CI: [{ci_lr[0]:.4f}, {ci_lr[1]:.4f}]")
print()

or_val = np.exp(coef_lr)
or_ci  = np.exp(ci_lr)
print(f"Odds Ratio                 : {or_val:.3f}")
print(f"  95% CI: [{or_ci[0]:.3f}, {or_ci[1]:.3f}]")
print()

# Average Partial Effect (APE) on the probability scale
# APE = E[P(Y=1|treated=1, X)] - E[P(Y=1|treated=0, X)]
# We use fit_naive for prediction (cluster SEs don't change point estimates)
mdf_t1 = model_df.copy(); mdf_t1['treated'] = 1
mdf_t0 = model_df.copy(); mdf_t0['treated'] = 0

ape_point = fit_naive.predict(mdf_t1).mean() - fit_naive.predict(mdf_t0).mean()
print(f"Average Partial Effect (APE): {ape_point:.4f}  ({ape_point*100:.2f} pp)")


=== Primary Logistic Regression (cluster-robust SEs) ===
Formula: hard_shell ~ treated + C(region) + C(season) + C(year) + C(sex) + total_soak_time
N = 163,758

Treatment coeff (log-odds) :  0.7309  SE=0.2446  p=0.0028
  95% CI: [0.2515, 1.2103]

Odds Ratio                 : 2.077
  95% CI: [1.286, 3.354]



Average Partial Effect (APE): 0.0660  (6.60 pp)


In [11]:
# ── Cluster bootstrap for APE (resample stations, not rows) ────────────
#
# We resample STATIONS with replacement because within-station observations
# are correlated (same location, gear, crew). Row-level bootstrap would treat
# correlated observations as independent and underestimate CI width.
#
# With only 11 unique stations, the bootstrap distribution will be coarser
# than with more clusters — this is a fundamental data limitation, not a
# methodological error. The resulting CI should be interpreted accordingly.

station_groups = {name: grp.reset_index(drop=True)
                  for name, grp in model_df.groupby('station name')}
clusters = np.array(list(station_groups.keys()))

print(f"Bootstrap setup:")
print(f"  {len(clusters)} unique stations  (n_clusters is the effective sample size for the bootstrap)")
print(f"  {len(model_df):,} total observations")
print()

n_boot   = 500
n_failed = 0
apes_boot = []

t_start = time.time()

b = 0
while b < n_boot:
    sampled = rng.choice(clusters, size=len(clusters), replace=True)
    boot_df = pd.concat([station_groups[c] for c in sampled], ignore_index=True)

    try:
        fit_b = smf.logit(formula, data=boot_df).fit(disp=0, maxiter=50)
        bt1 = boot_df.copy(); bt1['treated'] = 1
        bt0 = boot_df.copy(); bt0['treated'] = 0
        ape_b = fit_b.predict(bt1).mean() - fit_b.predict(bt0).mean()
        apes_boot.append(ape_b)
    except Exception:
        n_failed += 1

    b += 1

    # Timing check after 5th rep: reduce to 200 if each rep takes >30s
    if b == 5:
        elapsed_5 = time.time() - t_start
        per_rep = elapsed_5 / 5
        print(f"  First 5 reps: {elapsed_5:.1f}s total  ({per_rep:.1f}s/rep)")
        if per_rep > 30:
            n_boot = 200
            print(f"  ⚠  Reducing to {n_boot} replicates (>30s/rep threshold reached)")

total_time = time.time() - t_start
print(f"\nCompleted {len(apes_boot)} successful replicates "
      f"({n_failed} failed) in {total_time:.0f}s")

apes_boot = np.array(apes_boot)
ci_lower  = float(np.percentile(apes_boot, 2.5))
ci_upper  = float(np.percentile(apes_boot, 97.5))
ci_se     = float(apes_boot.std())

print(f"\n{'='*55}")
print(f"APE point estimate : {ape_point*100:.2f} pp")
print(f"Bootstrap 95% CI   : [{ci_lower*100:.2f},  {ci_upper*100:.2f}] pp")
print(f"Bootstrap SE       : {ci_se*100:.3f} pp")
print(f"{'='*55}")


Bootstrap setup:
  11 unique stations  (n_clusters is the effective sample size for the bootstrap)
  163,758 total observations



  First 5 reps: 9.4s total  (1.9s/rep)



Completed 490 successful replicates (10 failed) in 1006s

APE point estimate : 6.60 pp
Bootstrap 95% CI   : [-12.75,  17.59] pp
Bootstrap SE       : 6.739 pp


In [12]:
# Bootstrap APE distribution plot
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(apes_boot, bins=max(15, len(apes_boot)//10),
        color='#4575b4', edgecolor='white', alpha=0.8, density=True)
ax.axvline(ape_point, color='#d73027', linewidth=2.2,
           label=f'Point estimate: {ape_point*100:.2f} pp')
ax.axvline(ci_lower, color='#d73027', linewidth=1.6, linestyle='--',
           label=f'Bootstrap 95% CI: [{ci_lower*100:.2f}, {ci_upper*100:.2f}] pp')
ax.axvline(ci_upper, color='#d73027', linewidth=1.6, linestyle='--')
ax.axvline(0, color='black', linewidth=1.0, linestyle=':',
           label='APE = 0 (no effect)')
ax.set_xlabel('Average Partial Effect on Hard-Shell Probability (pp)')
ax.set_ylabel('Density')
ax.set_title(
    'Bootstrap Distribution of Depth Treatment APE\n'
    f'Station-clustered  |  {len(apes_boot)} replicates  |  n_clusters = {len(clusters)}'
)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('fig_bootstrap_ape.png', dpi=150, bbox_inches='tight')
plt.show()


**Interpretation**: The APE is the average change in hard-shell probability when moving a crab from deep to shallow water, averaged over the actual covariate distribution in the model frame. The sign tells us the direction: positive = shallow crabs are more likely to be hard-shell; negative = deep crabs are more likely.

The bootstrap CI with only 11 clusters will be **coarser** than a design with many independent stations. Because the bootstrap samples from 11 station-level "draws," the CI reflects station-level variance, not individual crab variance. This is the correct uncertainty to report for the causal question, but it means the CI will appear wide relative to the analytic cluster-robust CI. Both are honest — the bootstrap CI is more conservative and finite-sample valid.

If the CI excludes zero: there is statistical evidence of a depth-shell association after controlling for all included covariates. If it includes zero: the data are consistent with no adjusted effect at α = 0.05 within these two regions.


## Section 7: Sensitivity Analysis

Three complementary sensitivity analyses bound the reliability of the Section 6 estimate. Together they address: specification robustness (7.1), treatment effect heterogeneity (7.2), and unmeasured confounding (7.3).


### 7.1 Specification Curve

We fit the logistic regression under 8 nested specifications and plot the odds ratios with 95% CIs. **Specs 1–5** progressively add legitimate confounders; the OR should stabilize or converge as confounding is absorbed. **Specs 6–8** add post-treatment mediators (temperature, dissolved oxygen), labeled "BAD CONTROL." Any OR shift in specs 6–8 relative to spec 5 is empirical evidence of over-adjustment — the model is partialing out the mechanism under study.

All 8 specs are fit on the same complete-case dataset (restricting to rows with non-missing temp_c and do_mgl) so that sample size does not confound the comparison.


In [13]:
# Spec curve dataset: complete cases for ALL variables including env vars
spec_df = model_df.dropna(subset=['temp_c', 'do_mgl']).copy()
print(f"Spec curve dataset: {len(spec_df):,} rows (model_df minus missing temp/DO)")
print()

SPECS = [
    ('1. treated only',
     'hard_shell ~ treated'),
    ('2. + C(region)',
     'hard_shell ~ treated + C(region)'),
    ('3. + C(region) + C(season)',
     'hard_shell ~ treated + C(region) + C(season)'),
    ('4. + C(region) + C(season) + C(year)',
     'hard_shell ~ treated + C(region) + C(season) + C(year)'),
    ('5. Full primary',
     'hard_shell ~ treated + C(region) + C(season) + C(year) + C(sex) + total_soak_time'),
    ('6. Primary + temp_c  [BAD CONTROL]',
     'hard_shell ~ treated + C(region) + C(season) + C(year) + C(sex) + total_soak_time + temp_c'),
    ('7. Primary + do_mgl  [BAD CONTROL]',
     'hard_shell ~ treated + C(region) + C(season) + C(year) + C(sex) + total_soak_time + do_mgl'),
    ('8. Primary + temp_c + do_mgl  [BAD CONTROL]',
     'hard_shell ~ treated + C(region) + C(season) + C(year) + C(sex) + total_soak_time + temp_c + do_mgl'),
]

spec_rows = []
for label, fml in SPECS:
    try:
        fs = smf.logit(fml, data=spec_df).fit(disp=0, maxiter=200)
        coef = fs.params['treated']
        ci   = fs.conf_int().loc['treated']
        spec_rows.append({
            'spec': label,
            'log_odds': coef,
            'OR': np.exp(coef),
            'CI_lo': np.exp(ci[0]),
            'CI_hi': np.exp(ci[1]),
            'bad_control': 'BAD CONTROL' in label
        })
        print(f"{label}")
        print(f"  OR = {np.exp(coef):.3f}  "
              f"[{np.exp(ci[0]):.3f}, {np.exp(ci[1]):.3f}]")
    except Exception as e:
        print(f"{label}: FAILED ({e})")

spec_results = pd.DataFrame(spec_rows)


Spec curve dataset: 156,076 rows (model_df minus missing temp/DO)

1. treated only
  OR = 3.132  [2.668, 3.676]


2. + C(region)
  OR = 3.393  [2.889, 3.984]


3. + C(region) + C(season)
  OR = 3.241  [2.760, 3.806]


4. + C(region) + C(season) + C(year)
  OR = 1.923  [1.621, 2.280]


5. Full primary
  OR = 2.292  [1.926, 2.728]


6. Primary + temp_c  [BAD CONTROL]
  OR = 2.104  [1.763, 2.512]


7. Primary + do_mgl  [BAD CONTROL]
  OR = 2.263  [1.892, 2.707]


8. Primary + temp_c + do_mgl  [BAD CONTROL]
  OR = 2.137  [1.785, 2.560]


In [14]:
# Specification curve plot
fig, ax = plt.subplots(figsize=(12, 6))
y_pos   = list(range(len(spec_results)))
clrs    = ['#d73027' if bc else '#4575b4' for bc in spec_results['bad_control']]

for i, row in spec_results.iterrows():
    ax.plot([row['CI_lo'], row['CI_hi']], [i, i],
            color=clrs[i], linewidth=2.2, alpha=0.75, solid_capstyle='round')
    ax.scatter(row['OR'], i, color=clrs[i], s=70, zorder=5)

ax.axvline(1.0, color='black', linestyle='--', linewidth=1.1, label='OR = 1  (null)')
ax.set_yticks(y_pos)
ax.set_yticklabels(spec_results['spec'], fontsize=9)
ax.set_xlabel('Odds Ratio  (shallow vs. deep)  with 95% CI')
ax.set_title(
    'Specification Curve: Depth Treatment Effect on Hard-Shell Probability\n'
    'Specs 6–8 = BAD CONTROL  (post-treatment mediators added — demonstrates over-adjustment)'
)
blue_p = mpatches.Patch(color='#4575b4', label='Valid specification (specs 1–5)')
red_p  = mpatches.Patch(color='#d73027', label='BAD CONTROL — over-adjustment (specs 6–8)')
ax.legend(handles=[blue_p, red_p], loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig('fig_spec_curve.png', dpi=150, bbox_inches='tight')
plt.show()


**Interpretation**:

- **Specs 1 → 5 progression**: The unadjusted OR (Spec 1) is confounded by region and season. As we add region (Spec 2), season (Spec 3), year (Spec 4), sex and soak time (Spec 5), the OR shifts — confounding being absorbed. Spec 5 is the headline estimate.

- **Specs 6–8 (BAD CONTROL)**: Adding temperature and/or DO moves the OR further. This extra movement is **not** additional confounding control — it is over-adjustment bias. Depth physically determines temperature and DO (shallower water is warmer, lower DO), so conditioning on them absorbs part of the causal pathway from depth to shell condition. The OR "shrinks" (or changes) because the model is now comparing crabs at the same temperature/DO regardless of depth — which defeats the purpose of the analysis.

The fact that the OR shifts in specs 6–8 *empirically validates* the original capstone's concern: correlated predictors (depth, temp, DO) cannot all be included naïvely. This is what makes the primary model's exclusion of temp and DO defensible rather than arbitrary.


### 7.4 Sensitivity check: adjusting for maturity (carapace width)

Carapace width is a proxy for maturity and age. In the original predictive modeling phase, carapace width caused data leakage — the Random Forest learned a near-perfect shortcut (“big crab = hard shell”) that would not generalize to predicting shell condition for new, unmeasured crabs. Data leakage is a prediction concern, not a causal inference concern.

In the causal analysis we are not predicting shell condition for new crabs — we are estimating the effect of depth placement on existing observations. For this question, carapace width proxies maturity/age, which is a common cause of both depth preference (older crabs move deeper) and shell condition (older crabs have harder shells). Excluding it leaves maturity as an uncontrolled confounder.

However, including it carries a different risk: if depth also affects growth rate (warmer shallow water accelerates metabolism and growth), then carapace width sits on the causal pathway from depth to shell condition — making it a mediator as well as a maturity proxy. This is a genuinely hard case in causal inference: a post-treatment variable that also proxies for an unmeasured confounder.

The correct approach is to run both and report what changes:

- **Primary model (§6)**: excludes carapace width — estimates total effect of shallow placement including any maturity confounding
- **Sensitivity model (§7.4)**: includes carapace width — adjusts for maturity but risks partial over-adjustment if depth affects growth

The difference between the two ORs tells us how much of the observed association reflects maturity sorting vs. something else (depth-driven temperature exposure, habitat selection, or direct depth effects).

In [15]:
# Sensitivity check: add carapace_width as a 9th specification
# This adjusts for maturity/age as a confounder proxy
# Note: carapace_width has missing values — drop them for this model only

model_df_cw = model_df.dropna(subset=['carapace_width']).copy()
print(f"Rows with carapace_width: {len(model_df_cw):,} of {len(model_df):,} ({len(model_df_cw)/len(model_df)*100:.1f}%)")
print(f"Carapace width range: {model_df_cw['carapace_width'].min():.1f} \u2013 {model_df_cw['carapace_width'].max():.1f} mm")

formula_cw = formula + ' + carapace_width'

# Fit without carapace_width on same restricted sample (for fair comparison)
fit_no_cw = smf.logit(formula, data=model_df_cw).fit(
    disp=0, cov_type='cluster',
    cov_kwds={'groups': model_df_cw['station name']}
)

# Fit with carapace_width
fit_with_cw = smf.logit(formula_cw, data=model_df_cw).fit(
    disp=0, cov_type='cluster',
    cov_kwds={'groups': model_df_cw['station name']}
)

# Compare treatment coefficients
results_cw = []
for label, fit in [('Primary (no carapace width)', fit_no_cw),
                    ('Sensitivity (+ carapace width)', fit_with_cw)]:
    b = fit.params['treated']
    ci_l, ci_h = fit.conf_int().loc['treated']
    results_cw.append({
        'Model': label,
        'OR': np.exp(b),
        'CI_lo': np.exp(ci_l),
        'CI_hi': np.exp(ci_h),
        'log_OR': b
    })

cw_df = pd.DataFrame(results_cw)
print("\nTreatment effect \u2014 with vs without carapace width adjustment:")
print(cw_df.round(3).to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(7, 3))
y_pos = np.arange(len(cw_df))
ax.errorbar(
    cw_df['OR'], y_pos,
    xerr=[cw_df['OR'] - cw_df['CI_lo'], cw_df['CI_hi'] - cw_df['OR']],
    fmt='o', color='#4c72b0', capsize=5, markersize=8
)
ax.axvline(1.0, color='gray', linewidth=0.8, linestyle='--', label='OR = 1 (no effect)')
ax.set_yticks(y_pos)
ax.set_yticklabels(cw_df['Model'])
ax.set_xlabel('Odds ratio for shallow placement \u2192 hard shell')
ax.set_title('Sensitivity check: does adjusting for maturity (carapace width) change the finding?\n(If OR drops substantially, maturity confounding explains much of the association)')
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('fig_carapace_sensitivity.png', dpi=120)
plt.show()

# Interpret the change
or_change = cw_df.loc[1, 'OR'] - cw_df.loc[0, 'OR']
pct_change = (or_change / cw_df.loc[0, 'OR']) * 100
print(f"\nOR change from adding carapace width: {or_change:+.3f} ({pct_change:+.1f}%)")
if abs(pct_change) > 20:
    print(">> Substantial change \u2014 maturity confounding is a meaningful contributor to the observed association.")
    print("   The primary model's OR likely overstates the depth effect.")
elif abs(pct_change) > 10:
    print(">> Moderate change \u2014 maturity explains some but not all of the association.")
else:
    print(">> Small change \u2014 the depth association is robust to maturity adjustment.")
    print("   Maturity confounding does not fully explain the observed OR.")

Rows with carapace_width: 163,758 of 163,758 (100.0%)
Carapace width range: 112.0 – 209.0 mm



Treatment effect — with vs without carapace width adjustment:
                         Model    OR  CI_lo  CI_hi  log_OR
   Primary (no carapace width) 2.077  1.286  3.354   0.731
Sensitivity (+ carapace width) 1.471  0.782  2.764   0.386

OR change from adding carapace width: -0.606 (-29.2%)
>> Substantial change — maturity confounding is a meaningful contributor to the observed association.
   The primary model's OR likely overstates the depth effect.


### 7.2 Heterogeneous Treatment Effects (HTE) by Region

If depth affects shell condition through a universal mechanism (e.g., ambient temperature during soaking), treatment effects should be similar across regions. If Port Townsend Bay and Hood Canal show meaningfully different ORs, the pooled estimate in Section 6 masks region-specific dynamics, and WDFW should interpret the two regions separately.


In [16]:
# Within-region logistic regression (no region FE — within-region model)
formula_hte = ('hard_shell ~ treated + C(season) + C(year) + '
               'C(sex) + total_soak_time')

hte_rows = []
for region in ['Port Townsend Bay', 'Hood Canal']:
    rdf = model_df[model_df['region'] == region].copy()
    nt  = int(rdf['treated'].sum())
    nc  = int((rdf['treated'] == 0).sum())
    pb  = rdf['hard_shell'].mean()

    print(f"\n── {region} ─────────────────────────────────")
    print(f"  n={len(rdf):,}   treated={nt:,}   control={nc:,}   baseline={pb:.3f}")

    try:
        fs_r = smf.logit(formula_hte, data=rdf).fit(disp=0, maxiter=200)
        coef = fs_r.params['treated']
        ci   = fs_r.conf_int().loc['treated']
        or_r = np.exp(coef)

        r1 = rdf.copy(); r1['treated'] = 1
        r0 = rdf.copy(); r0['treated'] = 0
        ape_r = fs_r.predict(r1).mean() - fs_r.predict(r0).mean()

        print(f"  OR : {or_r:.3f}  [95% CI: {np.exp(ci[0]):.3f}, {np.exp(ci[1]):.3f}]")
        print(f"  APE: {ape_r:.4f}  ({ape_r*100:.2f} pp)")

        hte_rows.append({
            'Region': region,
            'n': len(rdf),
            'n_shallow': nt,
            'n_deep': nc,
            'OR': round(or_r, 3),
            'CI_lo': round(np.exp(ci[0]), 3),
            'CI_hi': round(np.exp(ci[1]), 3),
            'APE (pp)': round(ape_r * 100, 2)
        })
    except Exception as e:
        print(f"  FAILED: {e}")

print()
print(pd.DataFrame(hte_rows).to_string(index=False))



── Port Townsend Bay ─────────────────────────────────
  n=82,537   treated=2,845   control=79,692   baseline=0.845


  OR : 1.669  [95% CI: 1.392, 1.999]
  APE: 0.0488  (4.88 pp)

── Hood Canal ─────────────────────────────────
  n=81,221   treated=362   control=80,859   baseline=0.870


  OR : 11.039  [95% CI: 4.552, 26.773]
  APE: 0.1170  (11.70 pp)

           Region     n  n_shallow  n_deep     OR  CI_lo  CI_hi  APE (pp)
Port Townsend Bay 82537       2845   79692  1.669  1.392  1.999      4.88
       Hood Canal 81221        362   80859 11.039  4.552 26.773     11.70


**Interpretation**: If the two ORs / APEs differ substantially (say, by a factor of 2 or more), the pooled estimate from Section 6 is an average that may not represent either region well. Heterogeneous effects are ecologically plausible: Hood Canal is a semi-enclosed fjord with restricted tidal exchange and historically low DO, while Port Townsend Bay has stronger tidal flushing. Different baseline environmental conditions would produce different depth-response gradients even if the underlying mechanism were the same.

Note that within-region models have even fewer unique stations (Hood Canal has ~7 stations; Port Townsend Bay has ~4), so the region-specific ORs carry substantial uncertainty.


### 7.3 E-value for Unmeasured Confounding

The E-value (VanderWeele & Ding, 2017) quantifies how strong an **unmeasured confounder** would need to be to fully explain away the observed association. It asks: if there existed an unmeasured variable associated with both shallow depth (treatment) and hard-shell condition (outcome), how large would both associations need to be (on the risk-ratio scale) to shift the adjusted OR to 1.0?

Large E-value → robust to unmeasured confounders. Small E-value (< ~2) → even a modest confounder could overturn the finding.


In [17]:
def evalue(rr):
    # E-value (VanderWeele & Ding 2017). Flip so RR >= 1 before computing.
    if rr < 1:
        rr = 1.0 / rr
    return rr + np.sqrt(rr * (rr - 1))

# Use cluster-robust OR from Section 6 primary model
or_primary = float(np.exp(fit_cluster.params['treated']))
ci_lo_or   = float(np.exp(fit_cluster.conf_int().loc['treated', 0]))
ci_hi_or   = float(np.exp(fit_cluster.conf_int().loc['treated', 1]))

# CI bound closer to the null (OR=1)
or_ci_null = ci_lo_or if or_primary > 1 else ci_hi_or

ev_point = evalue(or_primary)
ev_ci    = evalue(or_ci_null)

print("E-value Analysis  (VanderWeele & Ding, 2017)")
print(f"{'─'*55}")
print(f"Primary OR (cluster-robust)  : {or_primary:.3f}")
print(f"  95% CI                     : [{ci_lo_or:.3f}, {ci_hi_or:.3f}]")
print(f"  CI bound closer to null    : {or_ci_null:.3f}")
print()
print(f"E-value (point estimate)     : {ev_point:.3f}")
print(f"E-value (CI bound)           : {ev_ci:.3f}")


E-value Analysis  (VanderWeele & Ding, 2017)
───────────────────────────────────────────────────────
Primary OR (cluster-robust)  : 2.077
  95% CI                     : [1.286, 3.354]
  CI bound closer to null    : 1.286

E-value (point estimate)     : 3.572
E-value (CI bound)           : 1.892


**Interpretation**: The E-value for the **point estimate** means an unmeasured confounder would need to be associated with *both* shallow-depth assignment AND hard-shell condition by a risk ratio of at least E-value (each) to fully explain away the observed association. The E-value for the **CI bound** is the more conservative figure — it tells us how strong confounding would need to be to push the CI entirely past the null.

For context: strong biological confounders in field ecology typically have risk ratios of 2–4. Station non-random placement (stations are at historically productive fishing grounds) is a plausible unmeasured confounder; its magnitude is unknown. If the E-value is ≥ 4, the finding is robust to all but the strongest confounders. If the E-value is < 2, the finding could be explained by moderate confounding, and WDFW should treat it with caution pending additional data.


## Section 8: Decision Framing

### What we can claim

- **The association is robust to observed confounders**: After controlling for region, season, year, sex, and soak time, the adjusted APE and its bootstrap CI characterize the depth–shell relationship within the Port Townsend Bay + Hood Canal common-support sample.
- **Region was the dominant source of confounding**: The SMD analysis (Section 3) showed region indicators as the most imbalanced covariates; restricting to PTB + Hood Canal (Section 4) substantially improved covariate balance.
- **Post-treatment over-adjustment is empirically demonstrated**: The specification curve (Section 7.1) shows that adding temperature and DO further changes the OR relative to the fully-adjusted primary model — providing direct empirical evidence of the bad-control problem the original capstone flagged.
- **Statistical resolution is not the binding constraint**: The MDE is well below 1 percentage point; if a substantial depth effect exists, this dataset has the power to find it.

### What we explicitly cannot claim

- **No inference for Marine Area 10 or Marine Area 13**: Shallow n < 50 in both regions; estimates there would be pure extrapolation beyond the common-support region.
- **No within-season causal inference**: The region × season coverage matrix (Section 2) shows no season in which all four regions are simultaneously sampled. Season and region effects are structurally confounded.
- **No mechanism claim**: The APE estimates the *total* depth effect on shell hardness, including any pathway through temperature and DO. We cannot decompose this into "direct" and "indirect" (mediated) effects without a separate mediation analysis or instrument.
- **No generalization beyond sampled stations**: Stations were placed at historically productive fishing grounds, not sampled randomly from all Puget Sound habitat. Estimates describe the 11 surveyed stations, not all PTB / Hood Canal habitat.
- **Hard-shell catch rates do not reflect population proportions** (outcome-dependent sampling): Pot surveys have outcome-dependent sampling: hard-shell crabs are active foragers more likely to enter traps, while soft-shell crabs hide post-molt in protective habitat and are systematically underrepresented in catches at all depths. The hard-shell catch rate in shallow traps is higher than in deep traps — but this reflects both true habitat association AND differential trap catchability by shell condition. Trawl surveys or telemetry data would be needed to separate these effects.
- **Depth does not cause shell hardness independent of maturity** (ontogenetic migration): Dungeness crabs exhibit ontogenetic depth migration — crabs in the 1\_2 hardening phase (58% of this dataset) biologically select shallow-to-mid habitat during post-molt hardening, where warmer temperatures accelerate calcium deposition. Maturity/age is therefore an unmeasured common cause of both depth preference and shell condition. The sensitivity check in §7.4 quantifies how much the OR changes when carapace width (a maturity proxy) is included. The E-value of 3.57 tells us how strong this unmeasured driver would need to be to fully explain the association — biological priors suggest it is plausible at that strength. The management implication (shallow deployments reliably catch proportionally more hard-shell crabs) holds regardless of mechanism, but the interpretation that depth itself drives shell hardness is not supported by this data alone.

### What WDFW would need to support stronger claims

- **Balanced sampling design**: Deploy shallow traps alongside deep traps within the same spatial cluster, in all four regions, across multiple seasons in the same year. This eliminates region × depth confounding by design — no covariate adjustment needed for region.
- **More shallow observations in MA10 and MA13**: Even 100–200 shallow records per region would enable the HTE analysis (Section 7.2) to cover all four regions and test whether the effect is region-specific or universal.
- **A natural experiment or instrument**: A historical management event or geographic feature that exogenously changes which depth strata are accessible at some stations (e.g., a tidal barrier, a new permit area) would allow causal identification without relying solely on covariate adjustment.

### Summary Table

| Question | Answer |
|---|---|
| Does depth (shallow vs. deep) predict hard-shell probability? | Yes, unadjusted — but heavily confounded by region |
| After adjustment, what is the estimated effect? | APE from §6 bootstrap (see final cell) |
| Which regions does this apply to? | **Port Townsend Bay and Hood Canal only** |
| Is the estimate robust to unmeasured confounders? | Quantified by E-value in §7.3 |
| Could we detect a 1 pp effect with this sample? | Yes — MDE is well below 1 pp |
| Why are temp/DO excluded from the primary model? | Post-treatment mediators (bad controls); spec curve §7.1 shows the over-adjustment empirically |
| Biggest remaining threat to validity? | Non-random station placement; only 11 stations creates coarse bootstrap uncertainty |


## Appendix: Method Notes

**Why logistic regression instead of Random Forest for this analysis?**
Random Forest optimizes predictive accuracy, not causal identification. Its parameters have no established causal interpretation, and there is no standard framework for producing E-values, specification curves, or cluster-robust CIs from a forest. Logistic regression's coefficient on `treated` is the log-odds difference conditional on all included covariates — interpretable as a causal quantity under the conditional ignorability assumption. For a causal analysis, interpretability and the ability to conduct sensitivity tests dominate raw predictive accuracy.

**Why cluster-bootstrap instead of analytic SEs?**
Within-station observations are not independent: same trap, same location, same survey crew, same tidal conditions. Analytic SEs assume independent observations and will systematically underestimate uncertainty. The cluster-robust sandwich SE corrects this asymptotically; the cluster-bootstrap provides finite-sample uncertainty without relying on large-sample normal approximations. With only 11 clusters, the bootstrap distribution will be coarse — this is an inherent limitation of the available data, and both the analytic cluster-robust SE and the bootstrap CI are reported.

**Why not propensity score matching (PSM)?**
PSM creates a pseudo-randomized comparison group by matching or reweighting on a propensity model. It has appealing properties but two practical problems here: (1) the common-support problem is severe — 87% of shallow observations come from one station, so matching would discard most deep observations; (2) PSM requires a correctly specified propensity model, which is no less assumption-laden than a correctly specified outcome model. Regression adjustment is more transparent for a first analysis. PSM is recommended as a robustness check in follow-up work.

**Why binary depth treatment instead of continuous depth?**
The depth distribution is bimodal (few observations in 30–80 ft), so a linear depth term would be driven by the extremes with poor interpolation across the mid range. The binary contrast is ecologically motivated (shallow tidal zone vs. deep survey stratum), interpretable without functional form assumptions, and consistent with the coarse sampling design. Future work with spline or GAM terms for depth — using only stations with observations at multiple depths — would be a natural extension.


## Final Summary (Resume Bullets)

In [18]:
print("=" * 60)
print("RESUME-READY NUMBERS")
print("=" * 60)
print(f"\n1. Adjusted Treatment APE (bootstrap 95% CI):")
print(f"   {ape_point*100:.2f} pp  [{ci_lower*100:.2f}, {ci_upper*100:.2f}] pp")
print(f"\n2. Imbalanced covariates |SMD| > 0.1 (before restriction):")
print(f"   {n_imbalanced}")
print(f"\n3. E-value (point / CI bound):")
print(f"   {ev_point:.3f} / {ev_ci:.3f}")
print("=" * 60)


RESUME-READY NUMBERS

1. Adjusted Treatment APE (bootstrap 95% CI):
   6.60 pp  [-12.75, 17.59] pp

2. Imbalanced covariates |SMD| > 0.1 (before restriction):
   11

3. E-value (point / CI bound):
   3.572 / 1.892
